In [ ]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog

import pandas as pd

## Understand data available for one specific player (Jalen Brunson), and begin with first stages of feature engineering

In [32]:
player_results = players.find_players_by_full_name("Jalen Brunson")
player_results

[{'id': 1628973,
  'full_name': 'Jalen Brunson',
  'first_name': 'Jalen',
  'last_name': 'Brunson',
  'is_active': True}]

In [33]:
brunson_id = player_results[0]["id"]

game_log = playergamelog.PlayerGameLog(
    player_id=brunson_id,
    season="2025-26"
)

brunson_games = game_log.get_data_frames()[0]

brunson_games.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,1628973,0022501176,"Apr 10, 2026",NYK vs. TOR,W,31,12,18,0.667,...,3,3,2,2,0,3,1,29,4,1
1,22025,1628973,0022501168,"Apr 09, 2026",NYK vs. BOS,W,37,10,19,0.526,...,1,1,10,0,0,1,3,25,8,1
2,22025,1628973,0022501143,"Apr 06, 2026",NYK @ ATL,W,39,11,26,0.423,...,3,3,13,2,0,3,1,30,7,1
3,22025,1628973,0022501123,"Apr 03, 2026",NYK vs. CHI,W,30,6,13,0.462,...,1,1,10,0,0,2,4,17,25,1
4,22025,1628973,0022501102,"Mar 31, 2026",NYK @ HOU,L,37,5,14,0.357,...,5,6,8,1,0,3,2,12,-26,1


In [34]:
brunson_games.shape

(74, 27)

In [ ]:
brunson_games.columns.tolist()

In [ ]:
brunson_games = brunson_games.sort_values("GAME_DATE")

brunson_games.head()

In [7]:
brunson_games["GAME_DATE"] = pd.to_datetime(brunson_games["GAME_DATE"])

In [8]:
brunson_games = brunson_games.sort_values("GAME_DATE").reset_index(drop=True)

In [ ]:
brunson_games["LAST_GAME_PTS"] = brunson_games["PTS"].shift(1)

brunson_games[
    ["GAME_DATE", "MATCHUP", "PTS", "LAST_GAME_PTS"]
].head(10)

In [35]:
for stat in ["PTS", "REB", "AST"]:
    brunson_games[f"{stat}_LAST_5_AVG"] = (
        brunson_games[stat]
        .shift(1)
        .rolling(window=5)
        .mean()
    )

    brunson_games[f"{stat}_LAST_10_AVG"] = (
        brunson_games[stat]
        .shift(1)
        .rolling(window=10)
        .mean()
    )

In [36]:
brunson_games[
    [
        "GAME_DATE",
        "PTS",
        "PTS_LAST_5_AVG",
        "PTS_LAST_10_AVG",
        "REB_LAST_5_AVG",
        "AST_LAST_5_AVG",
    ]
].head(15)

,GAME_DATE,PTS,PTS_LAST_5_AVG,PTS_LAST_10_AVG,REB_LAST_5_AVG,AST_LAST_5_AVG
0,"Apr 10, 2026",29,NaN,NaN,NaN,NaN
1,"Apr 09, 2026",25,NaN,NaN,NaN,NaN
2,"Apr 06, 2026",30,NaN,NaN,NaN,NaN
3,"Apr 03, 2026",17,NaN,NaN,NaN,NaN
4,"Mar 31, 2026",12,NaN,NaN,NaN,NaN
5,"Mar 29, 2026",32,22.6,NaN,2.8,8.6
6,"Mar 26, 2026",26,23.2,NaN,3.2,9.2
7,"Mar 24, 2026",32,23.4,NaN,3.4,9.8
8,"Mar 22, 2026",23,23.8,NaN,3.0,8.6
9,"Mar 20, 2026",17,25.0,NaN,2.8,7.4


## Breakdown top active players

In [37]:
from nba_api.stats.static import players

all_players = players.get_players()

len(all_players)

5135

In [ ]:
all_players[:5]

In [38]:
active_players = [player for player in all_players if player["is_active"]]

len(active_players)

571

In [39]:
active_players[:10]

[{'id': 1630173,
  'full_name': 'Precious Achiuwa',
  'first_name': 'Precious',
  'last_name': 'Achiuwa',
  'is_active': True},
 {'id': 203500,
  'full_name': 'Steven Adams',
  'first_name': 'Steven',
  'last_name': 'Adams',
  'is_active': True},
 {'id': 1628389,
  'full_name': 'Bam Adebayo',
  'first_name': 'Bam',
  'last_name': 'Adebayo',
  'is_active': True},
 {'id': 1630534,
  'full_name': 'Ochai Agbaji',
  'first_name': 'Ochai',
  'last_name': 'Agbaji',
  'is_active': True},
 {'id': 1630583,
  'full_name': 'Santi Aldama',
  'first_name': 'Santi',
  'last_name': 'Aldama',
  'is_active': True},
 {'id': 1641725,
  'full_name': 'Trey Alexander',
  'first_name': 'Trey',
  'last_name': 'Alexander',
  'is_active': True},
 {'id': 1629638,
  'full_name': 'Nickeil Alexander-Walker',
  'first_name': 'Nickeil',
  'last_name': 'Alexander-Walker',
  'is_active': True},
 {'id': 1628960,
  'full_name': 'Grayson Allen',
  'first_name': 'Grayson',
  'last_name': 'Allen',
  'is_active': True},
 {'id

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data_processing import get_player_game_logs

test_games = get_player_game_logs(
    player_id=1628973,
    season="2025-26"
)

test_games[["GAME_DATE", "MATCHUP", "PTS", "REB", "AST"]].head()

In [ ]:
from importlib import reload
import src.data_processing

reload(src.data_processing)

from src.data_processing import engineer_features

brunson_features = engineer_features(test_games)

brunson_features[
    [
        "GAME_DATE",
        "MATCHUP",
        "OPPONENT",
        "IS_HOME",
        "REST_DAYS",
        "PTS",
        "PTS_LAST_5_AVG",
        "PTS_LAST_10_AVG",
        "PTS_SEASON_AVG",
    ]
].head(12)

In [40]:
from importlib import reload
import src.data_processing

reload(src.data_processing)

from src.data_processing import build_player_dataset

brunson_dataset = build_player_dataset(
    player_id=1628973,
    season="2025-26"
)

brunson_dataset.head(12)

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,FG3A_LAST_10_AVG,FG3A_SEASON_AVG,FG3A_STD_LAST_5,FG3A_STD_LAST_10,FTA_LAST_GAME,FTA_LAST_5_AVG,FTA_LAST_10_AVG,FTA_SEASON_AVG,FTA_STD_LAST_5,FTA_STD_LAST_10
0,22025,1628973,0022500003,2025-10-22,NYK vs. CLE,W,34,5,18,0.278,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22025,1628973,0022500018,2025-10-24,NYK vs. BOS,W,35,10,20,0.500,...,9.000000,9.000000,NaN,NaN,13.0,13.000000,13.000000,13.000000,NaN,NaN
2,22025,1628973,0022500108,2025-10-26,NYK @ MIA,L,35,14,26,0.538,...,8.000000,8.000000,1.414214,1.414214,9.0,11.000000,11.000000,11.000000,2.828427,2.828427
3,22025,1628973,0022500125,2025-10-28,NYK @ MIL,L,35,14,25,0.560,...,9.000000,9.000000,2.000000,2.000000,4.0,8.666667,8.666667,8.666667,4.509250,4.509250
4,22025,1628973,0022500023,2025-10-31,NYK @ CHI,L,35,12,25,0.480,...,8.000000,8.000000,2.581989,2.581989,9.0,8.750000,8.750000,8.750000,3.685557,3.685557
5,22025,1628973,0022500153,2025-11-02,NYK vs. CHI,W,32,10,22,0.455,...,8.000000,8.000000,2.236068,2.236068,5.0,8.000000,8.000000,8.000000,3.605551,3.605551
6,22025,1628973,0022500159,2025-11-03,NYK vs. WAS,W,32,6,17,0.353,...,8.500000,8.500000,2.607681,2.345208,7.0,6.800000,7.833333,7.833333,2.280351,3.250641
7,22025,1628973,0022500175,2025-11-05,NYK vs. MIN,W,33,9,20,0.450,...,8.000000,8.000000,3.000000,2.516611,3.0,5.600000,7.142857,7.142857,2.408319,3.484660
8,22025,1628973,0022500192,2025-11-09,NYK vs. BKN,W,29,6,14,0.429,...,7.875000,7.875000,2.489980,2.356602,2.0,5.200000,6.500000,6.500000,2.863564,3.703280
9,22025,1628973,0022500208,2025-11-11,NYK vs. MEM,W,36,11,19,0.579,...,7.888889,7.888889,2.167948,2.204793,4.0,4.200000,6.222222,6.222222,1.923538,3.562926
